# 14d — LPT spherical density at 32³: MAP via `cadre.minimize`

Deterministic MAP optimization of the same posterior the samplers traverse, using `cadre.minimize` (unified optax/optimistix/scipy front-end; default `optax_lbfgs` with zoom line search). Three minimizations: **(1)** the field MAP at the *true* cosmology — exactly the inner-MAP problem MUSE solves per simulation (and the natural warm start to precompute on cluster); **(2)** the joint `(Ω_c, σ_8, δ_IC)` MAP for the pixel likelihood; **(3)** the same joint MAP for the harmonic likelihood.

Same config + mock (seed 0) as `14a`/`14b`/`14c`. Run headless with `uv run --no-sync papermill 14d-LPTDensityMAP.ipynb 14d-LPTDensityMAP.ipynb --cwd .`. Cost per minimization ≈ iterations × 2–4 gradient evaluations (line search) × the jitted gradient time from the gate below; the power-spectrum coloring makes the field problem ill-conditioned, so hundreds-to-thousands of L-BFGS iterations are normal.

In [ ]:
%load_ext autoreload
%autoreload 2
import os

os.environ["JAX_ENABLE_X64"] = "True"  # float32 => NaN/chaotic IC gradients
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")  # coexist with other GPU processes
# At 32^3 the pipeline is a chain of small sequential ops: a many-core CPU can beat a small GPU
# (measured: 0.14 s/grad on a 20-core CPU vs 0.51 s on an RTX 4060). Set JAX_PLATFORMS=cpu to force CPU.
os.environ.setdefault("JAX_PLATFORMS", "cuda,cpu")

import dataclasses
import time

import jax
import jax.numpy as jnp
import jax_cosmo as jc
import matplotlib.pyplot as plt
import numpy as np
import jax_fli as jfli
from jax.scipy.special import ndtr
from numpyro.handlers import condition, seed, trace

jax.config.update("jax_enable_x64", True)
from cadre import minimize as cadre_minimize

MESH = 32  # laptop/cluster-node size; bump for production or use the distributed 15-lensing-muse-inference.py
NSIDE = MESH
print(f"jax {jax.__version__}  backend {jax.default_backend()}  x64 {jax.config.jax_enable_x64}  MESH={MESH}")

## 1. Model configuration and mock observation

LPT-only (`sim_mode="lpt"`) → spherical galaxy overdensity (`lensing_output="density"`), two tomographic source planes at z = 0.10 and 0.17 inside the z ≤ 0.2 lightcone, centered observer (whole sky). The mock is a forward draw of the pixel model at a random prior point (seed 0 — identical across the 14a/14b/14c notebooks, so their posteriors are directly comparable); we condition on it and warm-start at the truth.

In [ ]:
cosmo = jc.Planck18()
box = tuple(float(x) for x in jfli.utils.compute_box_size_from_redshift(cosmo, 0.2, (0.5, 0.5, 0.5)))
priors = {
    "Omega_c": jfli.infer.PreconditionnedUniform(0.1, 0.5),
    "sigma8": jfli.infer.PreconditionnedUniform(0.6, 1.0),
}

config = jfli.ppl.Configurations(
    mesh_size=(MESH, MESH, MESH),
    box_size=box,
    halo_size=(0, 0),
    field_sharding=None,
    sim_mode="lpt",
    nbody_solver="BullFrog",
    t0=0.001,
    t1=1.0,
    lpt_order=1,
    number_of_shells=5,
    nb_steps=5,
    paint_order="cic",
    gradient_order=4,
    laplace_fd=True,
    shell_spacing="a",
    time_stepping="D",
    min_width=1.0,
    lensing_output="density",
    map2alm_method="jax",
    likelihood_space="pixel",
    min_redshift=0.001,
    max_redshift=0.2,
    n_integrate=8,
    nside=NSIDE,
    geometry="spherical",
    scheme="rbf_neighbor",
    observer_position=(0.5, 0.5, 0.5),
    paint_nside=NSIDE,
    kernel_width_pixels=0.8,
    fiducial_cosmology=jc.Planck18,
    nz_shear=[0.10, 0.17],
    priors=priors,
    sigma_e=0.3,
    adjoint="checkpointed",
    checkpoints=2,
)

pixel_model = jfli.ppl.full_field_probmodel(config)
tr = trace(seed(pixel_model, 0)).get_trace()
x_obs = jnp.stack([v["value"] for k, v in tr.items() if "observable" in k and k != "observable_meta_data"], axis=0)
x_obs_meta_data = tr["observable_meta_data"]["value"]
theta_truth = jnp.array([float(tr["Omega_c_base"]["value"]), float(tr["sigma8_base"]["value"])])
Oc_true, s8_true = float(tr["Omega_c"]["value"]), float(tr["sigma8"]["value"])
truth = {"Omega_c": Oc_true, "sigma8": s8_true}
print(f"truth: Omega_c={Oc_true:.4f}  sigma8={s8_true:.4f}   observable {x_obs.shape}")

# Harmonic variant of the same config: NEVER mutate the shared dataclass in place — use dataclasses.replace.
ell_max, taper = min(2 * NSIDE - 1, 3 * NSIDE // 2), 4
config_h = dataclasses.replace(config, likelihood_space="harmonic", ell_max=int(ell_max), ell_taper_width=taper)
harmonic_model = jfli.ppl.full_field_probmodel(config_h, observed_maps=x_obs)

In [ ]:
jfli.SphericalDensity.FromDensityMetadata(array=x_obs, field=x_obs_meta_data).show()

## 2. Sanity gate: jitted log-density + gradient at the truth

Expect **large** cosmology-base gradients (O(10²–10³) at 32³): at the true parameters the score is a zero-mean random variable with std = √Fisher, and a field-level likelihood has a big Fisher information for 2 parameters. This is expected statistics, not a normalization bug (no noise variance depends on the sampled cosmology) — see `docs/WORK_IN_PROGRESS/20-likelihood-audit.md`. Time gradients **under `jax.jit`**: an eager `jax.grad` call is dispatch-bound and ~50× slower at this size (the historical "8 s per gradient" was that artifact).

In [ ]:
data = {f"observable_{i}": x_obs[i] for i in range(x_obs.shape[0])}
cond_model = condition(pixel_model, data=data)

from numpyro.infer.util import initialize_model

init, potential_fn, postprocess_fn, model_trace = initialize_model(jax.random.key(0), cond_model)
logdensity_fn = lambda position: -potential_fn(position)
value_and_grad_fn = jax.jit(jax.value_and_grad(logdensity_fn))

truth_position = {
    "Omega_c_base": theta_truth[0],
    "sigma8_base": theta_truth[1],
    "initial_conditions": tr["initial_conditions"]["value"].array,
}
t0 = time.time()
val, grad = jax.block_until_ready(value_and_grad_fn(truth_position))
print(f"compile+first-run: {time.time() - t0:.1f} s")
t0 = time.time()
for _ in range(3):
    val, grad = jax.block_until_ready(value_and_grad_fn(truth_position))
print(f"jitted gradient: {(time.time() - t0) / 3 * 1e3:.0f} ms")
assert np.isfinite(float(val)) and all(bool(np.all(np.isfinite(np.asarray(g)))) for g in jax.tree.leaves(grad))
print(f"logp={float(val):.1f}  dOc_base={float(grad['Omega_c_base']):.3e}  ds8_base={float(grad['sigma8_base']):.3e}")

## 3. MAP: field-only at the truth, then joint (pixel), then joint (harmonic)

Two caveats that matter. First, never initialize the white IC at exactly zero — a zero white field puts every particle exactly on the mesh, where the mass-assignment gradient has a kink and the gradient comes back NaN; seed with `1e-2 · N(0, I)`. Second, the cosmology at the **joint** MAP is *not* a marginal cosmology estimate (no volume factors — that is what MUSE in `14a` provides); the joint MAP is a geometry/optimizer gate, checked by `logp(MAP) ≥ logp(truth)`.

In [ ]:
# (1) Field MAP at the TRUE cosmology — the MUSE inner-MAP problem.
data_ic = {f"observable_{i}": x_obs[i] for i in range(x_obs.shape[0])}
data_ic["Omega_c_base"] = theta_truth[0]
data_ic["sigma8_base"] = theta_truth[1]
ic_model = condition(pixel_model, data=data_ic)
_, potential_ic, _, _ = initialize_model(jax.random.key(0), ic_model)

z0 = 1e-2 * jax.random.normal(jax.random.PRNGKey(3), config.mesh_size)  # NEVER exactly zero (painting kink)

t0 = time.time()
ic_map, st_ic = cadre_minimize(
    potential_ic,
    {"initial_conditions": z0},
    solver_name="optax_lbfgs",
    max_iter=3000,
    rtol=1e-8,
    atol=1e-8,
    record_history=True,
    options={"max_linesearch_steps": 50},
)
g_ic = jax.jit(jax.grad(potential_ic))(ic_map)["initial_conditions"]
print(f"IC-only MAP: {time.time() - t0:.0f}s, {int(st_ic.iter_num)} iters, "
      f"loss={float(st_ic.best_loss):.2f}, |grad|={float(jnp.linalg.norm(g_ic)):.2e}")

ic_true = np.asarray(tr["initial_conditions"]["value"].array).ravel()
ic_hat = np.asarray(ic_map["initial_conditions"]).ravel()
print(f"corr(IC_MAP, IC_truth) = {float(np.corrcoef(ic_true, ic_hat)[0, 1]):.3f}   "
      f"std ratio = {ic_hat.std() / ic_true.std():.3f}   (Wiener-like shrinkage: both < 1)")

In [ ]:
# (2) JOINT MAP (Omega_c, sigma8, IC) — pixel likelihood. Gate: logp(MAP) >= logp(truth).
init_map = {"Omega_c_base": jnp.array(0.0), "sigma8_base": jnp.array(0.0), "initial_conditions": z0}

t0 = time.time()
map_params, st = cadre_minimize(
    potential_fn,
    init_map,
    solver_name="optax_lbfgs",
    max_iter=3000,
    rtol=1e-8,
    atol=1e-8,
    record_history=True,
    options={"max_linesearch_steps": 50},
)
print(f"joint MAP (pixel): {time.time() - t0:.0f}s, {int(st.iter_num)} iters")
print(f"logp(MAP) = {-float(st.best_loss):.2f}   vs   logp(truth) = {float(val):.2f}")
assert -float(st.best_loss) >= float(val) - 1e-3, "joint MAP failed to reach at least the truth's logp"
for n, p in priors.items():
    phys = float(p.low + (p.high - p.low) * ndtr(map_params[f"{n}_base"]))
    print(f"{n}: joint-MAP={phys:.4f}   truth={truth[n]:.4f}   (joint MAP != marginal estimate; see 14a)")

In [ ]:
# (3) JOINT MAP — harmonic likelihood (data already bound via observed_maps -> no condition()).
_, potential_h, _, _ = initialize_model(jax.random.key(0), harmonic_model)

t0 = time.time()
map_h, st_h = cadre_minimize(
    potential_h,
    init_map,
    solver_name="optax_lbfgs",
    max_iter=3000,
    rtol=1e-8,
    atol=1e-8,
    options={"max_linesearch_steps": 50},
)
print(f"joint MAP (harmonic): {time.time() - t0:.0f}s, {int(st_h.iter_num)} iters, loss={float(st_h.best_loss):.2f}")
for n, p in priors.items():
    phys = float(p.low + (p.high - p.low) * ndtr(map_h[f"{n}_base"]))
    print(f"{n}: harmonic joint-MAP={phys:.4f}   truth={truth[n]:.4f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for state_, label in ((st_ic, "IC-only @ true cosmo"), (st, "joint pixel")):
    h = np.asarray(state_.history)[:, : int(state_.iter_num)]
    ax[0].semilogy(h[0], label=label)
    ax[1].plot(h[1], label=label)
ax[0].set(xlabel="L-BFGS iteration", ylabel="|step|")
ax[1].set(xlabel="L-BFGS iteration", ylabel="objective (pre-step)")
ax[0].legend()
plt.tight_layout()
plt.show()

## Methods note

`cadre.minimize` wraps the solver zoo behind one call; `optax_lbfgs` here matches the L-BFGS MUSE uses for its inner MAP (`jax_fli.infer.muse._z_map`), so cell (1) is a standalone version of that solve — its converged `|grad|` is the `map_gnorm` MUSE gates on, and its wall time calibrates `map_maxiter` for `14a`. Warm-starting from a previous MAP (pass it as `init_params`) is the field-scale speedup: MUSE re-solves this problem per simulation per Newton step and inherits each previous solution. For bounded or preconditioned variants, `cadre.minimize` takes `lower_bound`/`upper_bound`/`precondition=True`, and `solver_name="scipy_l-bfgs-b"` runs the scipy path.